# Step 2: Java Source Code Translation (Layer 2 & 3 Consolidated)

This notebook simulates and prototypes the code refactoring process on the `working/sonar-stash` project codebase.

This step focuses on:
- Detecting JDK 17 deprecated/removed APIs (Layer 2) using `jdeprscan`.
- Collecting compilation errors under JDK 17 (Layer 3) using `mvn compile`.
- Bundling findings by file and querying local Ollama LLM to refactor using a list of non-overlapping block edits applied from bottom to top.

In [9]:
import sys
import os
import json
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root added to path: {PROJECT_ROOT}")

# Check availability of key imports
from src.tools.maven import MavenProject, MavenPomEditor, MavenRunner
from src.core.tree_sitter_engine import UniversalASTReader
from src.tools.codebase_search_tools import find_code_usages, search_codebase

print("Successfully imported models and tools!")

# Setup paths globally (Adjust PROJECT_PATH to point to your target Java project)
PROJECT_PATH = Path("working/sonar-stash").resolve()
pom_path = PROJECT_PATH / "pom.xml"
ARTIFACTS_DIR = PROJECT_PATH / "artifacts"

review_file = ARTIFACTS_DIR / "reader_review.json"
if not review_file.exists():
    review_file = Path("dependency_focus_scopes.json").resolve()

print(f"Working Project path: {PROJECT_PATH} (exists: {PROJECT_PATH.exists()})")
print(f"Artifacts path: {ARTIFACTS_DIR}")
print(f"Review report path: {review_file} (exists: {review_file.exists()})")

Project root added to path: D:\capstone_project\MYGRATE---Capstone-Project
Successfully imported models and tools!
Working Project path: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash (exists: True)
Artifacts path: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash\artifacts
Review report path: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash\artifacts\reader_review.json (exists: True)


In [10]:
# Load POM upgrade results from POM migration phase
review_data = {}
selected_sol = {}

if review_file.exists():
    with open(review_file, "r", encoding="utf-8") as f:
        review_data = json.load(f)
    review_body = review_data.get("review_upgrade_candidates", {})
    selected_sol = review_body.get("selected_solution", {})
    print("Upgraded Dependency Mappings (from reader_review.json):")
    print(json.dumps(selected_sol, indent=2))
else:
    print("reader_review.json not found!")

Upgraded Dependency Mappings (from reader_review.json):
{
  "org.sonarsource.sonarqube:sonar-plugin-api": "9.3.0.51899",
  "com.github.cliftonlabs:json-simple": "4.0.1",
  "org.asynchttpclient:async-http-client": "3.0.2",
  "com.google.guava:guava": "20.0",
  "org.assertj:assertj-core": "3.26.3",
  "org.mockito:mockito-core": "5.15.2",
  "org.mockito:mockito-junit-jupiter": "5.15.2",
  "org.awaitility:awaitility": "4.1.1",
  "com.github.tomakehurst:wiremock": "2.27.2",
  "org.picocontainer:picocontainer": "2.14.2",
  "com.google.code.gson:gson": "2.12.0"
}


## 1. Consolidated Compilation & Deprecated API Scan

We run JDK 17 compilation to get modern compiler errors and execute `jdeprscan` to capture deprecated/removed API usages. These are aggregated by Java source file.

In [11]:
from src.tools.maven import MavenRunner, MavenProject
from src.tools.jdeprscan_pipeline import run_jdeprscan_pipeline
import re

# 1. Compile project under JDK 17 once to extract initial compile errors
print("Compiling project under JDK 17...")
runner = MavenRunner(target_java_version="17")
compile_res = runner.compile(PROJECT_PATH, clean=True)
print(f"Initial compile status: {compile_res.status}")

# 2. Run jdeprscan pipeline under JDK 17 to detect deprecated/removed APIs (Level 2)
print("Running jdeprscan pipeline on the working codebase...")
updated_jdeprscan_result = run_jdeprscan_pipeline(
    project_path=str(PROJECT_PATH),
    target_release="17",
    logger=lambda msg: print(f"  {msg}")
)

file_findings = {}

# Regex pattern to parse maven javac error details
# Matches: src/main/java/pkg/File.java:3: error: ...
#      or: src\main\java\pkg\File.java:[3,1] error: ...
err_pattern = r'(src[/\\]main[/\\]java[/\\]\S+?\.java)[:\[](\d+)(?:,\d+)?\]?:?\s+error:\s+(.*)'

# Parse errors from JDK 17 compilation
compile_errors_matched = re.findall(err_pattern, compile_res.stdout) + re.findall(err_pattern, compile_res.stderr)
print(f"\nExtracted {len(compile_errors_matched)} compile error(s) under JDK 17:")
for rel_path, line_no, err_msg in compile_errors_matched:
    standard_path = rel_path.replace("\\", "/")
    file_findings.setdefault(standard_path, {"findings": [], "matching_rules": []})
    file_findings[standard_path]["findings"].append(f"Compile Error (Line {line_no}): {err_msg}")
    print(f"  * {standard_path}:{line_no} -> {err_msg}")

# Parse compile errors from jdeprscan b1 step (runs JDK 8 compile as part of scan pipeline)
b1_step = updated_jdeprscan_result.get("steps", {}).get("b1_compile", {})
if b1_step.get("status") == "FAIL":
    b1_error = b1_step.get("error", "")
    b1_matches = re.findall(err_pattern, b1_error)
    for rel_path, line_no, err_msg in b1_matches:
        standard_path = rel_path.replace("\\", "/")
        file_findings.setdefault(standard_path, {"findings": [], "matching_rules": []})
        file_findings[standard_path]["findings"].append(f"Compile Error (Line {line_no}): {err_msg}")

# Parse jdeprscan B2 project scan warnings/removal items
b2_step = updated_jdeprscan_result.get("steps", {}).get("b2_project_scan", {})
for line in b2_step.get("for_removal", []) + b2_step.get("deprecated", []):
    m = re.match(r"class\s+([^\s]+)\s+uses", line)
    if m:
        class_path = m.group(1)
        rel_path = f"src/main/java/{class_path}.java"
        file_findings.setdefault(rel_path, {"findings": [], "matching_rules": []})
        file_findings[rel_path]["findings"].append(f"Jdeprscan warning: {line}")

# Parse reader review flagged risks
if review_file.exists():
    review_body = review_data.get("review_upgrade_candidates", {})
    risks_text = review_body.get("risks", "")
    candidates = re.findall(r"\b(?:javax\.\w+|com\.sun\.\w+)\b", risks_text)
    for cand in candidates:
        matches = search_codebase(
            project_path=str(PROJECT_PATH),
            regex_pattern=r"import\s+" + cand.replace(".", r"\."),
            file_extensions=["java"]
        )
        for m in matches:
            rel_path = m["file_path"].replace("\\", "/")
            file_findings.setdefault(rel_path, {"findings": [], "matching_rules": []})
            file_findings[rel_path]["findings"].append(f"Reader review flagged risk: usage of {cand}")

# Deduplicate findings per file
for path, info in file_findings.items():
    info["findings"] = list(set(info["findings"]))

# Load rules and match them against findings and codebase contents
rules_file = Path("migration_rules.json")
rules = []
if rules_file.exists():
    with open(rules_file, "r", encoding="utf-8") as f:
        rules = json.load(f).get("rules", [])

for rel_path, info in file_findings.items():
    full_path = PROJECT_PATH / rel_path
    if not full_path.exists():
        continue
    file_content = full_path.read_text(errors="ignore")
    for rule in rules:
        target = rule.get("target")
        if target:
            is_match = False
            for fnd in info["findings"]:
                if target in fnd:
                    is_match = True
                    break
            if not is_match:
                if rule.get("type") == "import_declaration":
                    if f"import {target}" in file_content:
                        is_match = True
                else:
                    target_short = target.split()[-1].split("(")[0]
                    if target_short in file_content:
                        is_match = True
            if is_match and rule not in info["matching_rules"]:
                info["matching_rules"].append(rule)

print(f"\nFound {len(file_findings)} file(s) requiring translation:")
for path, info in file_findings.items():
    print(f"\n- File: {path}")
    print(f"  Findings:")
    for fnd in info["findings"]:
        print(f"    * {fnd}")
    print(f"  Matching Rules:")
    for r in info["matching_rules"]:
        print(f"    * [{r.get('rule_id')}] {r.get('target')} -> {r.get('new')} ({r.get('reason')})")

# Save diagnostic reports to target/ and artifacts/ so the agent can load them
for folder in ["artifacts", "target"]:
    dir_path = PROJECT_PATH / folder
    dir_path.mkdir(parents=True, exist_ok=True)
    with open(dir_path / "jdeprscan_report.json", "w", encoding="utf-8") as f:
        json.dump(updated_jdeprscan_result, f, indent=2, default=str)
    change_candidates = []
    for f_path, f_info in file_findings.items():
        change_candidates.append({
            "file_path": f_path,
            "reason": "; ".join(f_info["findings"]),
            "matching_rules": f_info["matching_rules"]
        })
    mygrate_report = {
        "status": "ok",
        "change_candidates": change_candidates
    }
    with open(dir_path / "mygrate_report.json", "w", encoding="utf-8") as f:
        json.dump(mygrate_report, f, indent=2, default=str)
print("\nSuccessfully saved jdeprscan_report.json and mygrate_report.json to target/ and artifacts/!")

Compiling project under JDK 17...
Initial compile status: 1
Running jdeprscan pipeline on the working codebase...
  [jdeprscan] Khám phá JDK...
  [jdeprscan] JDK 8:  C:\Users\tngtr\AppData\Local\Java\jdk-8
  [jdeprscan] JDK 17: C:\Program Files\Java\jdk-17
  [jdeprscan] jdeprscan: C:\Program Files\Java\jdk-17\bin\jdeprscan.exe
  [jdeprscan] Maven: C:\Users\tngtr\AppData\Local\Apache\apache-maven-3.9.16\bin\mvn
  [jdeprscan] B0: Maven resolve dependencies...
  [jdeprscan] B0: đang resolve dependencies...
  [jdeprscan] B0: Maven FAIL (exit code 1)
  [jdeprscan] B1: Compile project...
  [jdeprscan] B1: compiling 27 source files...
  [jdeprscan] B1: Compile FAIL — D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash\src\main\java\org\sonar\plugins\stash\StashIssueReportingPostJob.java:3: error: package com.google.common.annotations does not exist

  [jdeprscan] B2: jdeprscan project JAR...
  [jdeprscan] B2: project JAR không có — skip
  [jdeprscan] B3: jdeprscan dependencies.

## 2. Iterative Translation Loop

We run a focused code translation loop to resolve JDK 17 migration blockers file-by-file.
Here we define our `TranslatorAgent_2` agent directly in the notebook to make it easily editable and debuggable.

In [ ]:
import os
import json
import re
from pathlib import Path
from typing import Any
from datetime import datetime
from langchain_core.messages import ToolMessage
from src.agents.base_agent import BaseAgent, ToolDefinition
from src.tools.maven import MavenRunner

class TranslatorAgent_2(BaseAgent):
    """
    TranslatorAgent_2 — batch-edit migration agent with logging.

    Workflow:
        1. read_file   →  inspect current source (with line numbers)
        2. apply_edits →  submit ALL changes as line-based edits (NO auto-compile)
        3. compile_project →  compile once to verify all changes
        4. (if errors) →  apply_edits again to fix, then compile again
    """

    # Keep more file content in history so agent doesn't forget what it read.
    READ_FILE_KEEP_MESSAGES = 6

    def __init__(self, model_name: str | None = None):
        super().__init__(model_name)
        self.project_path = ""
        self.current_file = ""  # track which file is being migrated
        self.MAX_ITERATIONS = 15
        self._log_entries: list[str] = []
        self._last_edit_count = 0  # track edits since last compile

    def get_system_prompt(self) -> str:
        return (
            "You are an expert Java migration agent specializing in upgrading projects to JDK 17.\n"
            "Your task is to resolve compiler errors, removed JDK APIs, and deprecations in the target file.\n\n"
            "Follow these rules strictly:\n"
            "1. First, call `read_file` to inspect the file (output includes line numbers).\n"
            "2. Analyze ALL findings, rules, and compile errors. Plan ALL changes at once.\n"
            "3. Call `apply_edits` ONCE with ALL changes. This tool does NOT compile.\n"
            "4. After applying all edits, call `compile_project` to compile the project and check for errors.\n"
            "5. If the compile result shows remaining errors, read the relevant files, apply fixes with `apply_edits`, then `compile_project` again.\n"
            "6. Do NOT call `compile_project` twice in a row without making edits in between — "
            "if the previous compile showed errors, you MUST call `apply_edits` before compiling again.\n"
            "7. Do NOT call write_file. Prefer `apply_edits`.\n"
            "8. You must call tools to perform edits. Do not output code blocks directly.\n\n"
            "IMPORTANT — Migration quality rules:\n"
            "- Do NOT comment out imports. Find the correct replacement API or add a TODO comment explaining what needs to change.\n"
            "- Do NOT replace type references with `Object`. Find the actual replacement type.\n"
            "- Do NOT uncomment or modify Maven plugin versions unless specifically asked. "
            "If a plugin resolution error occurs, comment out the offending plugin in pom.xml.\n"
            "- When a class or method has moved packages in the new API version, find and use the correct new package path.\n"
            "- Prefer real fixes over workarounds like `// TODO`. Only use `// TODO` when a genuine replacement API cannot be determined.\n\n"
            "Keep your responses concise. When you are finished and the project compiles, state that you have completed the migration.\n"
        )

    def run(self, instruction: str) -> str:
        payload = self._parse_instruction(instruction)
        self.project_path = payload.get("project_path", "")
        self._log_entries = []
        self._last_edit_count = 0
        return super().run(instruction)

    def _log(self, entry: str):
        self._log_entries.append(entry)

    def get_log(self) -> str:
        return "\n".join(self._log_entries)

    def _invoke_with_retry(
        self,
        llm_with_tools: Any,
        messages: list,
        tool_map: dict[str, ToolDefinition],
        payload: dict[str, Any],
        react_tool_results: dict[str, Any],
    ) -> Any | None:
        keep = self.READ_FILE_KEEP_MESSAGES
        for i, msg in enumerate(messages):
            if not isinstance(msg, ToolMessage):
                continue
            messages_after = len(messages) - i - 1
            if messages_after <= keep:
                continue
            if i == 0:
                continue
            prev_msg = messages[i - 1]
            tool_calls = getattr(prev_msg, "tool_calls", [])
            is_read_file = False
            for tc in tool_calls:
                if tc.get("id") == msg.tool_call_id and tc.get("name") == "read_file":
                    is_read_file = True
                    break
            if is_read_file:
                messages[i] = ToolMessage(
                    content="[File content was shown earlier. Re-call read_file if you need to see it again.]",
                    tool_call_id=msg.tool_call_id,
                )
        return super()._invoke_with_retry(llm_with_tools, messages, tool_map, payload, react_tool_results)

    def get_tools(self) -> list[ToolDefinition]:
        return [
            ToolDefinition(
                name="read_file",
                description="Read the full content of a file (with line numbers). Use this first to inspect the file's current state.",
                func=self._tool_read_file,
                parameters={
                    "type": "object",
                    "properties": {
                        "file_path": {
                            "type": "string",
                            "description": "Path to the Java source file relative to project root",
                        }
                    },
                    "required": ["file_path"]
                }
            ),
            ToolDefinition(
                name="apply_edits",
                description=(
                    "Apply multiple line-based edits to a file in a single call. Does NOT compile. "
                    "Each edit specifies start_line and end_line (1-based, inclusive) and the replacement text. "
                    "Edits are applied bottom-to-top so line numbers stay valid. "
                    "After calling this, you MUST call compile_project to verify your changes."
                ),
                func=self._tool_apply_edits,
                parameters={
                    "type": "object",
                    "properties": {
                        "file_path": {
                            "type": "string",
                            "description": "Path to the Java source file relative to project root",
                        },
                        "edits": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "start_line": {
                                        "type": "integer",
                                        "description": "Start line number to replace (1-based, inclusive)",
                                    },
                                    "end_line": {
                                        "type": "integer",
                                        "description": "End line number to replace (1-based, inclusive). Same as start_line for single-line edit.",
                                    },
                                    "replacement": {
                                        "type": "string",
                                        "description": "The replacement text for the line range. Use \\n for newlines.",
                                    }
                                },
                                "required": ["start_line", "end_line", "replacement"]
                            },
                            "description": "Array of edits. Each edit replaces lines start_line through end_line (inclusive) with replacement text. Order does not matter — edits are sorted and applied bottom-to-top."
                        }
                    },
                    "required": ["file_path", "edits"]
                }
            ),
            ToolDefinition(
                name="compile_project",
                description=(
                    "Compile the project using Maven under JDK 17. Call this AFTER applying edits to verify your changes. "
                    "Returns ALL compile errors (not filtered by file) so you can see which files need fixing."
                ),
                func=self._tool_compile_project,
                parameters={
                    "type": "object",
                    "properties": {},
                    "required": []
                }
            ),
        ]

    # ── Tool implementations ──

    def _resolve_path(self, file_path: str) -> Path:
        """Resolve a file path to an absolute Path object."""
        path = Path(file_path)
        if not path.is_absolute():
            path = Path(self.project_path) / path
        return path

    def _tool_read_file(self, file_path: str, **kwargs) -> str:
        try:
            path = self._resolve_path(file_path)
            if not path.exists():
                self._log(f"[read_file] ERROR: File {file_path} does not exist.")
                return f"Error: File {file_path} does not exist."
            self.current_file = str(path)
            content = path.read_text(encoding="utf-8")
            lines = content.splitlines()
            numbered = [f"{i+1:4d} | {line}" for i, line in enumerate(lines)]
            self._log(f"[read_file] {file_path} ({len(lines)} lines)")
            return "\n".join(numbered)
        except Exception as e:
            self._log(f"[read_file] ERROR: {e}")
            return f"Error reading file {file_path}: {e}"

    def _tool_apply_edits(self, file_path: str, edits: list, **kwargs) -> str:
        """Apply multiple line-based edits, sorted bottom-to-top. Does NOT compile."""
        try:
            path = self._resolve_path(file_path)
            if not path.exists():
                return f"Error: File {file_path} does not exist."
            self.current_file = str(path)

            if not edits:
                return "Error: No edits provided."

            content = path.read_text(encoding="utf-8")
            lines = content.splitlines()

            validated = []
            for edit in edits:
                start = int(edit.get("start_line", 0))
                end = int(edit.get("end_line", start))
                replacement = edit.get("replacement", "")
                # Convert literal \n (backslash + n) to actual newlines.
                # LLMs sometimes pass \n as a literal two-char sequence instead of a newline.
                if "\\n" in replacement and "\n" not in replacement:
                    replacement = replacement.replace("\\n", "\n")
                if start < 1 or end < 1 or start > end:
                    return f"Error: Invalid line range start_line={start}, end_line={end}."
                if end > len(lines):
                    return f"Error: end_line={end} exceeds file length ({len(lines)} lines)."
                validated.append((start, end, replacement))

            validated.sort(key=lambda x: x[0], reverse=True)  # bottom-to-top

            results = []
            for start, end, replacement in validated:
                original = "\n".join(lines[start - 1:end])
                new_lines = replacement.splitlines() if replacement else []
                lines[start - 1:end] = new_lines
                results.append(f"Lines {start}-{end}: {len(original)} chars → {len(replacement)} chars")
                self._log(f"[apply_edits] Lines {start}-{end} replaced")

            new_content = "\n".join(lines)
            path.write_text(new_content, encoding="utf-8")
            self._last_edit_count += len(validated)
            self._log(f"[apply_edits] {len(validated)} edit(s) applied to {file_path} (no compile yet)")
            return f"Applied {len(validated)} edit(s) to {file_path}. Call compile_project to verify.\n" + "\n".join(results)

        except Exception as e:
            self._log(f"[apply_edits] ERROR: {e}")
            return f"Error applying edits to {file_path}: {e}"

    def _tool_compile_project(self, **kwargs) -> dict[str, Any]:
        """Compile project under JDK 17 and return ALL compile errors (not filtered by file)."""
        try:
            # Guard: warn if calling compile without having made any edits since last compile
            if self._last_edit_count == 0:
                self._log(f"[compile_project] WARNING: called without any edits since last compile")

            project_path = Path(self.project_path)  # ensure Path, not str
            runner = MavenRunner(target_java_version="17")
            compile_res = runner.compile(project_path, clean=True)
            output = compile_res.stdout + "\n" + compile_res.stderr
            status = compile_res.status

            # Reset edit counter after compile
            self._last_edit_count = 0

            if status == 0:
                self._log(f"[compile_project] exit_code=0, SUCCESS")
                return {
                    "exit_code": 0,
                    "success": True,
                    "errors": "Project compiles successfully! No errors."
                }

            # Extract ALL error lines — don't filter by file
            all_errors = self._extract_all_errors(output)
            self._log(f"[compile_project] exit_code={status}, errors={len(all_errors.splitlines())} line(s)")

            return {
                "exit_code": status,
                "success": False,
                "errors": all_errors if all_errors else "Compilation failed but no specific error lines found. Full output:\n" + output[-2000:]
            }
        except Exception as e:
            self._log(f"[compile_project] ERROR: {e}")
            return {"error": f"Failed to run compilation: {e}"}

    def _extract_all_errors(self, compile_output: str) -> str:
        """Extract ALL error lines from Maven compile output, including detail lines.

        Maven prints errors in blocks like:
            [ERROR] /path/File.java:[25,52] cannot find symbol
            [ERROR]   symbol:   class PostJob
            [ERROR]   location: class org.sonar.plugins.stash.StashIssueReportingPostJob

        We include all [ERROR] lines so the agent sees the full context (symbol name, location, etc.).
        """
        lines = compile_output.split("\n")
        # Include ALL lines that are part of error output:
        # - Lines with [ERROR] (includes both primary errors and detail lines like "symbol:", "location:")
        # - Lines with .java: and error: (for non-[ERROR]-prefixed formats)
        error_lines = []
        for line in lines:
            if "[ERROR]" in line or (".java" in line and "error:" in line.lower()):
                error_lines.append(line)

        if not error_lines:
            return ""
        # Return up to 120 error lines — enough for the LLM to see full context
        return "\n".join(error_lines[:120])

print("TranslatorAgent_2 class defined successfully!")

In [ ]:
import json
import re
from pathlib import Path
from datetime import datetime

LOG_FILE = Path("log.txt")

# ── PHASE 1: Blind fix — apply changes to each file without compiling ──
print("=" * 80)
print("PHASE 1: Blind Fix — apply changes per file (no compile)")
print("=" * 80)

phase1_results = {}

for rel_path, info in file_findings.items():
    file_path = PROJECT_PATH / rel_path
    if not file_path.exists():
        print(f"[WARNING] File does not exist: {file_path}")
        continue

    print(f"\n--- Processing: {rel_path} ---")

    findings_str = "\n".join([f"- {f}" for f in info["findings"]])
    rules_str = "\n".join([
        f"- [{r.get('rule_id')}] Replace '{r.get('target')}' with '{r.get('new')}' (Reason: {r.get('reason')})"
        for r in info["matching_rules"]
    ])

    payload = {
        "project_path": str(PROJECT_PATH),
        "target_java_version": "17",
        "current_instruction": f"Migrate the file {rel_path} to JDK 17, resolving all deprecations and compile errors. Apply all changes using apply_edits. Do NOT call compile_project — just apply the edits.",
        "findings": findings_str,
        "rules": rules_str
    }

    agent = TranslatorAgent_2()
    result = agent.run(json.dumps(payload, ensure_ascii=False, indent=2))

    # Log phase 1
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"\n{'='*80}\n")
        f.write(f"PHASE 1 — FILE: {rel_path}\n")
        f.write(f"TIME: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"{'='*80}\n\n")
        f.write("--- Findings ---\n")
        f.write(findings_str + "\n\n")
        f.write("--- Matching Rules ---\n")
        f.write(rules_str + "\n\n")
        f.write("--- Tool Call Log ---\n")
        f.write(agent.get_log() + "\n\n")
        f.write("--- Final Result ---\n")
        f.write(str(result)[:3000] + "\n")
        f.write(f"\n---\n\n")

    phase1_results[rel_path] = result
    print(f"  → Phase 1 complete for {rel_path}")

print(f"\n✓ Phase 1 complete. Processed {len(phase1_results)} file(s).")

# ── PHASE 2: Whole-project compile ──
print("\n" + "=" * 80)
print("PHASE 2: Whole-Project Compile")
print("=" * 80)

from src.tools.maven import MavenRunner
runner = MavenRunner(target_java_version="17")
compile_res = runner.compile(PROJECT_PATH, clean=True)
compile_output = compile_res.stdout + "\n" + compile_res.stderr

print(f"\nCompile exit code: {compile_res.status}")
if compile_res.status != 0:
    error_lines = [line for line in compile_output.split("\n")
                   if "[ERROR]" in line or "error:" in line]
    print(f"Found {len(error_lines)} error line(s):")
    for line in error_lines[:50]:
        print(f"  {line}")
else:
    print("✓ Project compiles successfully!")

# Log phase 2
with open(LOG_FILE, "a", encoding="utf-8") as f:
    f.write(f"\n{'='*80}\n")
    f.write(f"PHASE 2 — WHOLE-PROJECT COMPILE\n")
    f.write(f"TIME: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"{'='*80}\n\n")
    f.write(f"Exit code: {compile_res.status}\n\n")
    if compile_res.status != 0:
        f.write("--- Compile Output ---\n")
        f.write(compile_output[:5000] + "\n")
    else:
        f.write("✓ Project compiles successfully!\n")
    f.write(f"\n---\n\n")

# ── PHASE 3: Iterative fix — per file with compile errors ──
if compile_res.status != 0:
    print("\n" + "=" * 80)
    print("PHASE 3: Iterative Fix — fix each file with errors, then verify")
    print("=" * 80)

    # Parse compile errors to find which files have problems
    err_pattern = r'((?:src[/\\][\w./-]+\.java)|[A-Z]:[/\\][\w./\\-]+\.java)[:\[](\d+)(?:,\d+)?\]?:?\s+(?:error:\s+)?(.+)'
    errors_by_file = {}

    for match in re.finditer(err_pattern, compile_output):
        filepath = match.group(1).replace("\\", "/")
        line_no = match.group(2)
        err_msg = match.group(3)
        errors_by_file.setdefault(filepath, []).append(f"Compile Error (Line {line_no}): {err_msg}")

    # Deduplicate
    for f, errs in errors_by_file.items():
        errors_by_file[f] = list(set(errs))

    print(f"\nFiles with compile errors: {list(errors_by_file.keys())}")
    for f, errs in errors_by_file.items():
        print(f"  {f}: {len(errs)} error(s)")

    phase3_results = {}

    for rel_path_err, errs in errors_by_file.items():
        file_path = PROJECT_PATH / rel_path_err
        if not file_path.exists():
            print(f"[WARNING] File does not exist: {file_path}")
            continue

        print(f"\n--- Fixing: {rel_path_err} ({len(errs)} error(s)) ---")

        findings_str = "\n".join([f"- {e}" for e in errs])
        payload = {
            "project_path": str(PROJECT_PATH),
            "target_java_version": "17",
            "current_instruction": (
                f"Fix the compile errors in {rel_path_err}. "
                f"Read the file, apply all fixes using apply_edits, then call compile_project to verify. "
                f"If the project still has errors in OTHER files after your fix, that's OK — just ensure this file is correct."
            ),
            "findings": findings_str,
            "rules": ""
        }

        agent = TranslatorAgent_2()
        result = agent.run(json.dumps(payload, ensure_ascii=False, indent=2))

        # Log phase 3
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(f"\n{'='*80}\n")
            f.write(f"PHASE 3 — FIX: {rel_path_err}\n")
            f.write(f"TIME: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"{'='*80}\n\n")
            f.write("--- Compile Errors ---\n")
            f.write(findings_str + "\n\n")
            f.write("--- Tool Call Log ---\n")
            f.write(agent.get_log() + "\n\n")
            f.write("--- Final Result ---\n")
            f.write(str(result)[:3000] + "\n")
            f.write(f"\n---\n\n")

        phase3_results[rel_path_err] = result
        print(f"  → Phase 3 complete for {rel_path_err}")

    # Final verification compile
    print("\n" + "=" * 80)
    print("FINAL VERIFICATION: Whole-Project Compile")
    print("=" * 80)

    final_res = runner.compile(PROJECT_PATH, clean=True)
    print(f"\nFinal compile exit code: {final_res.status}")
    if final_res.status == 0:
        print("✓ Project compiles successfully after all fixes!")
    else:
        final_output = final_res.stdout + "\n" + final_res.stderr
        final_errors = [line for line in final_output.split("\n")
                        if "[ERROR]" in line or "error:" in line]
        print(f"  {len(final_errors)} error(s) remain:")
        for line in final_errors[:30]:
            print(f"  {line}")

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"\n{'='*80}\n")
        f.write(f"FINAL VERIFICATION\n")
        f.write(f"TIME: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"{'='*80}\n\n")
        f.write(f"Exit code: {final_res.status}\n")
        if final_res.status != 0:
            f.write("\n--- Remaining Errors ---\n")
            f.write("\n".join(final_errors[:50]) + "\n")
        else:
            f.write("✓ Project compiles successfully!\n")

print(f"\n{'='*80}")
print(f"TRANSLATION COMPLETE. Full log at: {LOG_FILE.resolve()}")
print(f"{'='*80}")

## 3. Final Verification & Secondary Checks

We verify that the whole codebase compiles successfully under JDK 17, and check remaining AST structures for libraries like Mockito.

In [ ]:
# Run a final clean compilation with MavenRunner targeting Java 17 to verify all changes
runner = MavenRunner(target_java_version="17")
print("Running final clean mvn compile to verify project state...")
res = runner.compile(PROJECT_PATH, clean=True)
print(f"Compilation exit code: {res.status}")

if res.status == 0:
    print("Success! The project compiles cleanly under JDK 17 after translation and upgrades.")
else:
    print("Errors encountered during compile. Showing stdout/stderr output:")
    print("--- STDOUT ---")
    print(res.stdout)
    print("--- STDERR ---")
    print(res.stderr)

Running final clean mvn compile to verify project state...
Compilation exit code: 1
Errors encountered during compile. Showing stdout/stderr output:
--- STDOUT ---
[INFO] Scanning for projects...
[INFO] 
[INFO] --------------------< org.sonar:sonar-stash-plugin >--------------------
[INFO] Building Stash 1.7.0-SNAPSHOT
[INFO]   from pom.xml
[INFO] ----------------------------[ sonar-plugin ]----------------------------
[WARNING] Ignoring incompatible plugin version 4.0.0-beta-2: 
The plugin org.apache.maven.plugins:maven-install-plugin:4.0.0-beta-2 has unmet prerequisites: 
	Required Maven version 4.0.0-rc-2 is not met by current version 3.9.16
[INFO] Latest version of plugin org.apache.maven.plugins:maven-install-plugin failed compatibility check
[INFO] Looking for compatible RELEASE version of plugin org.apache.maven.plugins:maven-install-plugin
[WARNING] Ignoring incompatible plugin version 4.0.0-beta-1: 
The plugin org.apache.maven.plugins:maven-install-plugin:4.0.0-beta-1 has unme

In [ ]:
# Find usages of Mockito in import statements using tree-sitter AST queries in the isolated directory
print("Querying AST for Mockito import declarations:")
mockito_imports = find_code_usages(
    project_path=str(PROJECT_PATH),
    node_type="import_declaration",
    identifier="mockito"
)

print(f"Found {len(mockito_imports)} Mockito import nodes:")
for imp in mockito_imports[:10]:
    print(f"  - {imp['file_path']} | Line {imp['start_line']}: {imp['snippet'].strip()}")

Querying AST for Mockito import declarations:
Found 44 Mockito import nodes:
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 5: import static org.mockito.ArgumentMatchers.any;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 6: import static org.mockito.ArgumentMatchers.anyString;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 7: import static org.mockito.ArgumentMatchers.eq;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 8: import static org.mockito.Mockito.mock;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 9: import static org.mockito.Mockito.times;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 10: import static org.mockito.Mockito.verify;
  - src\test\java\org\sonar\plugins\stash\StashIssueReportingPostJobTest.java | Line 11: import static org.mockito.Mockito

In [ ]:
print("=== STEP 2 MIGRATION COMPLETION SUMMARY ===")
print(f"Code translation and JDK 17 migration completed in isolated folder!")
print(f"The migrated codebase is preserved at: {PROJECT_PATH}")

=== STEP 2 MIGRATION COMPLETION SUMMARY ===
Code translation and JDK 17 migration completed in isolated folder!
The migrated codebase is preserved at: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash
